# ⚡ Vid2Rig — GVHMR Main Engine on Free Kaggle GPU

Turn any video into 3D motion capture on rigged Mixamo characters powered by **GVHMR (Main 3D Engine)** on Kaggle GPU, with optional MediaPipe live preview.

### ⚙️ Kaggle Notebook Settings:
1. **Accelerator**: Select **GPU T4 x2** or **GPU P100** (Free 30 hrs/week)
2. **Internet**: Toggle **Internet ON** in the right-hand panel (Settings → Internet → ON)
3. Run the cells below in order!

### Step 1: Verify NVIDIA GPU
Confirms your Tesla T4 / P100 GPU is allocated and ready for GVHMR acceleration.

In [ ]:
!nvidia-smi

### Step 2: Install System Dependencies & Cloudflare Tunnel
Installs Node.js, ffmpeg, and downloads the official `cloudflared` Linux binary.

In [ ]:
!apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

### Step 3: Setup Vid2Rig Repository & GVHMR Main Engine Dependencies
Clones Vid2Rig, installs web server dependencies, installs the GVHMR GPU pipeline, and downloads pretrained weights.

In [ ]:
import os
import sys

# Clone repo if not already in directory
if not os.path.exists("server.mjs"):
    !git clone https://github.com/liladhiee-web/vid2rig.git
    %cd vid2rig

# 1. Install Node.js web server dependencies
!npm install --omit=dev
!mkdir -p uploads takes hq/weights hq/body_models/smplx

# 2. Set PATH & upgrade build tooling
os.environ["PATH"] = f"/opt/conda/bin:{os.path.expanduser('~/.local/bin')}:{os.environ.get('PATH', '')}"
!pip install -q --upgrade pip setuptools wheel "numpy>=1.26"

# 3. Pre-install chumpy with build isolation bypassed (fixes Python 3.10/3.11 PEP 517 build failure)
!pip install -q --no-build-isolation "git+https://github.com/mattloper/chumpy@9b045ff5d6588a24a0bab52c83f032e2ba433e17" || pip install -q chumpy-fix || pip install -q --no-build-isolation "chumpy==0.70"

# 4. Install GVHMR Main Engine dependencies
!pip install -q gvhmr==1.6.1 ultralytics fastapi uvicorn roma mediapipe
!pip install -q --no-deps "git+https://github.com/warmshao/WiLoR-mini" || true
!pip install -q "git+https://github.com/MaxLiu22/Motion2MixamoRig@2835dcdfba30e78a2610360b9b2a1e1121d85159" || true

# 5. Download SMPL-X Neutral Body Model (required for 3D body motion)
!wget -q -nc https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz -O hq/body_models/smplx/SMPLX_NEUTRAL.npz

# 6. Set GVHMR model paths & download pretrained weights
os.environ["GVHMR_CHECKPOINTS"] = os.path.abspath("hq/weights")
os.environ["GVHMR_BODY_MODELS"] = os.path.abspath("hq/body_models")
!gvhmr download demo || python -m gvhmr.cli.demo download || python -c "import gvhmr; print('GVHMR successfully verified')" || true

### Step 4: Launch Unified Stack (GVHMR GPU Engine + Web Server + Cloudflare Tunnel)
This starts the GVHMR Main GPU Solver on port 5178, the Web Server on port 5177, and creates a secure Cloudflare Tunnel.
Click the generated **`https://*.trycloudflare.com`** URL to open Vid2Rig Studio!

In [ ]:
import subprocess
import time
import re
import urllib.request
import json
from IPython.display import HTML, display

# 1. Start GVHMR Main Engine GPU Worker on port 5178
print("Starting GVHMR Main Engine GPU Worker on port 5178...")
worker_log = open("/tmp/vid2rig_worker.log", "w")
worker_proc = subprocess.Popen(["python", "hq/worker.py"], stdout=worker_log, stderr=subprocess.STDOUT)
time.sleep(4)

# 2. Start Node.js Web Server on port 5177
print("Starting Vid2Rig Web Server on port 5177...")
server_log = open("/tmp/vid2rig_server.log", "w")
server_proc = subprocess.Popen(["node", "server.mjs"], stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(2)

# Check GVHMR worker status
try:
    with urllib.request.urlopen("http://127.0.0.1:5178/health", timeout=3) as resp:
        h = json.loads(resp.read().decode())
        print(f"✅ GVHMR Worker Status: {h}")
except Exception:
    print("⏳ GVHMR Worker is loading model pipeline (check /tmp/vid2rig_worker.log if needed)")

# 3. Start Cloudflare Tunnel
print("Starting Cloudflare Tunnel to port 5177...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5177", "--no-autoupdate"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None

for line in tunnel_proc.stdout:
    match = url_pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    display(HTML(f"""
    <div style="background:#0a0f0d;border:2px solid #00ff88;padding:22px;border-radius:8px;margin:15px 0;box-shadow:0 0 30px rgba(0,255,136,0.35);font-family:sans-serif;">
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:8px;">
            <span style="background:#00ff88;color:#000;font-weight:900;font-size:11px;padding:3px 8px;border-radius:3px;letter-spacing:0.5px;">GVHMR MAIN ENGINE ACTIVE</span>
            <span style="color:#00ff88;font-size:12px;font-family:monospace;">GPU ACCELERATED</span>
        </div>
        <h2 style="color:#fff;margin:0 0 8px 0;letter-spacing:0.5px;">⚡ Vid2Rig Studio is Live!</h2>
        <p style="color:#c2d8cd;margin:0 0 16px 0;font-size:14px;">Your Kaggle GPU session is running the full GVHMR 3D SMPL motion solver and web studio:</p>
        <a href="{tunnel_url}" target="_blank" style="background:#00ff88;color:#000;font-weight:bold;font-size:15px;padding:12px 26px;text-decoration:none;border-radius:4px;display:inline-block;letter-spacing:0.5px;">
            🚀 OPEN VID2RIG STUDIO &rarr;
        </a>
        <p style="color:#769a87;font-size:12px;margin:14px 0 0 0;font-family:monospace;">Direct Link: <a href="{tunnel_url}" target="_blank" style="color:#00ff88;">{tunnel_url}</a></p>
    </div>
    """))
    print(f"\n>>> Public URL: {tunnel_url}\n")
else:
    print("Cloudflare tunnel failed to produce a public URL. Check /tmp/vid2rig_worker.log and /tmp/vid2rig_server.log")

# Keep running
try:
    for line in tunnel_proc.stdout:
        pass
except KeyboardInterrupt:
    print("Stopping worker, server, and tunnel...")
    worker_proc.terminate()
    server_proc.terminate()
    tunnel_proc.terminate()
